# Patent Phrase-to-Phrase Matching — Complete Google Colab Notebook

**Combined notebook:** this merges the pretrained-weight download step from `download_weights.ipynb` into the starter project and completes the required work: baseline, controlled experiments, validation/error analysis, and submission file creation.

> The uploaded files did not include `project_brief.md`. This notebook therefore completes the tasks explicitly listed in the starter notebook. The original instructions describe a Kaggle competition; Colab is used here for development and generating `submission.csv`. Upload the final notebook to Kaggle separately if the competition requires a Kaggle notebook submission.

## 1. Runtime and installation

In Colab, choose **Runtime → Change runtime type → T4 GPU** if available. This notebook can run on CPU, but model fine-tuning will be much slower.

In [ ]:
!pip -q install -U transformers datasets accelerate sentencepiece huggingface_hub scikit-learn pandas scipy

In [ ]:
import os, glob, gc, random, warnings, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
from datasets import Dataset
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import Ridge
from sklearn.metrics.pairwise import cosine_similarity
from tokenizers.processors import TemplateProcessing
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    DataCollatorWithPadding, Trainer, TrainingArguments
)
from huggingface_hub import snapshot_download

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
transformers.set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch:", torch.__version__, "| Transformers:", transformers.__version__, "| Device:", device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 2. Download pretrained weights (combined from `download_weights.ipynb`)

This downloads the model once into the Colab runtime. Unlike the Kaggle offline workflow, Colab needs internet access for this first download. The downloaded checkpoint is then loaded from the local folder.

In [ ]:
MODEL_ID = "microsoft/deberta-v3-small"
MODEL_DIR = "/content/deberta-v3-small"

if not os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=MODEL_DIR,
        allow_patterns=["*.json", "*.safetensors", "*.bin", "*.model", "*.txt"],
    )
print("Checkpoint ready:", MODEL_DIR)

## 3. Load competition data

Upload `train.csv` and `test.csv` from the Kaggle competition dataset. If you have the competition data in Google Drive, mount Drive and set `DATA_DIR` to its folder instead.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Select train.csv and test.csv

DATA_DIR = "/content"
train_path = os.path.join(DATA_DIR, "train.csv")
test_path = os.path.join(DATA_DIR, "test.csv")
assert os.path.exists(train_path) and os.path.exists(test_path), (
    "Please upload both train.csv and test.csv."
)

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
print("Train shape:", train.shape, "| Test shape:", test.shape)
display(train.head())
print("\nScore distribution:")
display(train["score"].value_counts().sort_index())

## 4. Configuration and context preparation

The model compares the anchor phrase with the target phrase plus a short description of the CPC section. The CPC section is the first character of the context code.

In [ ]:
MAX_LEN = 64
EPOCHS = 3
LR = 2e-5
BATCH_SIZE = 16  # conservative for a Colab T4; increase if GPU memory permits
VAL_FRACTION = 0.20
OUTPUT_DIR = "/content/patent_phrase_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CPC_SECTIONS = {
    "A": "human necessities",
    "B": "performing operations; transporting",
    "C": "chemistry; metallurgy",
    "D": "textiles; paper",
    "E": "fixed constructions",
    "F": "mechanical engineering; lighting; heating; weapons; blasting",
    "G": "physics",
    "H": "electricity",
}

required = {"id", "anchor", "target", "context"}
assert required.issubset(train.columns), f"Train must contain {required}"
assert required.issubset(test.columns), f"Test must contain {required}"
assert "score" in train.columns, "train.csv must contain the score column"

def add_inputs(df):
    df = df.copy()
    df["text_a"] = df["anchor"].fillna("").astype(str)
    section = df["context"].fillna("").astype(str).str[0]
    section_text = section.map(CPC_SECTIONS).fillna("patent technology")
    df["text_b"] = df["target"].fillna("").astype(str) + "; " + section_text
    return df

train = add_inputs(train)
test = add_inputs(test)
print(train[["text_a", "text_b", "score"]].head())

## 5. Validation split — hold out complete anchors

A random row split can leak the same anchor into both training and validation sets. `GroupShuffleSplit` keeps all rows for an anchor in only one split. Reuse this exact split for every experiment.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=VAL_FRACTION, random_state=SEED)
train_idx, val_idx = next(splitter.split(train, groups=train["anchor"]))
trn = train.iloc[train_idx].reset_index(drop=True)
val = train.iloc[val_idx].reset_index(drop=True)
assert set(trn.anchor).isdisjoint(set(val.anchor))
print(f"Training rows: {len(trn)} | Validation rows: {len(val)}")
print("Distinct validation anchors:", val.anchor.nunique())

## 6. Baseline without fine-tuning

The baseline uses TF-IDF word and character n-grams with cosine similarity. It does not train or fine-tune a Transformer. Pearson correlation is the competition metric.

In [ ]:
def safe_pearson(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=float).reshape(-1)
    if len(y_true) < 2 or np.std(y_true) == 0 or np.std(y_pred) == 0:
        return float("nan")
    return float(np.corrcoef(y_true, y_pred)[0, 1])

# Similarity between anchor and target/context text.
word_vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)
char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=1, sublinear_tf=True)

baseline_text = pd.concat([trn["text_a"], trn["text_b"], val["text_a"], val["text_b"]], ignore_index=True)
word_vec.fit(baseline_text)
char_vec.fit(baseline_text)

def tfidf_pair_similarity(frame):
    a_word = word_vec.transform(frame["text_a"])
    b_word = word_vec.transform(frame["text_b"])
    a_char = char_vec.transform(frame["text_a"])
    b_char = char_vec.transform(frame["text_b"])
    word_sim = np.asarray(a_word.multiply(b_word).sum(axis=1)).ravel()
    char_sim = np.asarray(a_char.multiply(b_char).sum(axis=1)).ravel()
    return 0.6 * word_sim + 0.4 * char_sim

baseline_pred = tfidf_pair_similarity(val)
baseline_pearson = safe_pearson(val["score"], baseline_pred)
print(f"TF-IDF baseline validation Pearson: {baseline_pearson:.4f}")
baseline_results = [{"experiment": "TF-IDF word+character baseline", "pearson": baseline_pearson}]

## 7. Tokenization

The tokenizer explicitly receives a pair of text segments: `[CLS] anchor [SEP] target; context [SEP]`. Dynamic padding is applied per batch.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
tokenizer.backend_tokenizer.post_processor = TemplateProcessing(
    single="[CLS] $A [SEP]",
    pair="[CLS] $A [SEP] $B:1 [SEP]:1",
    special_tokens=[
        ("[CLS]", tokenizer.cls_token_id),
        ("[SEP]", tokenizer.sep_token_id),
    ],
)

def to_dataset(frame, max_len=MAX_LEN, with_labels=True):
    columns = ["text_a", "text_b"] + (["score"] if with_labels else [])
    work = frame[columns].copy()
    if with_labels:
        work["score"] = work["score"].astype("float32")
    ds = Dataset.from_pandas(work, preserve_index=False)
    ds = ds.map(
        lambda batch: tokenizer(
            batch["text_a"], batch["text_b"],
            truncation=True, max_length=max_len
        ),
        batched=True,
        remove_columns=["text_a", "text_b"],
    )
    if with_labels:
        ds = ds.rename_column("score", "labels")
    return ds

ds_trn = to_dataset(trn)
ds_val = to_dataset(val)
ds_test = to_dataset(test, with_labels=False)
print(tokenizer.decode(ds_trn[0]["input_ids"]))

## 8. Fine-tuning function

A single output neuron predicts a continuous similarity score, so this is a regression task trained with mean squared error. Evaluation reports Pearson correlation.

In [ ]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    return {"pearson": safe_pearson(labels, preds)}

def train_cross_encoder(max_len=64, learning_rate=2e-5, epochs=3,
                        batch_size=16, run_name="main"):
    train_ds = to_dataset(trn, max_len=max_len)
    valid_ds = to_dataset(val, max_len=max_len)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_DIR, num_labels=1, dtype=torch.float32
    )
    args = TrainingArguments(
        output_dir=f"{OUTPUT_DIR}/{run_name}",
        num_train_epochs=epochs,
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        warmup_ratio=0.1,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=100,
        fp16=torch.cuda.is_available(),
        report_to="none",
        seed=SEED,
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics,
    )
    start = time.time()
    trainer.train()
    elapsed = time.time() - start
    predictions = trainer.predict(valid_ds).predictions.reshape(-1)
    score = safe_pearson(val["score"], predictions)
    print(f"{run_name}: Pearson={score:.4f}; training time={elapsed/60:.1f} min")
    return trainer, predictions, score, elapsed

## 9. Main fine-tuned model

This is the primary model configuration. If the runtime disconnects, reduce `EPOCHS` or use a smaller batch size, then rerun from the setup cells.

In [ ]:
trainer, main_val_pred, main_pearson, main_seconds = train_cross_encoder(
    max_len=MAX_LEN,
    learning_rate=LR,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    run_name="main_model",
)
baseline_results.append({"experiment": "DeBERTa-v3-small fine-tuned (main)", "pearson": main_pearson})

## 10. Controlled experiments

Each experiment changes **one setting at a time** relative to the main model. These runs require additional GPU time. Run both for the assignment report; if Colab time is limited, run them in separate sessions while preserving the same train/validation split.

In [ ]:
# Experiment 1: change only maximum sequence length (64 -> 96).
exp1_trainer, exp1_pred, exp1_score, exp1_seconds = train_cross_encoder(
    max_len=96, learning_rate=LR, epochs=EPOCHS,
    batch_size=BATCH_SIZE, run_name="experiment_maxlen_96"
)
baseline_results.append({"experiment": "Experiment 1: max length 96", "pearson": exp1_score})

In [ ]:
# Experiment 2: change only learning rate (2e-5 -> 1e-5).
exp2_trainer, exp2_pred, exp2_score, exp2_seconds = train_cross_encoder(
    max_len=MAX_LEN, learning_rate=1e-5, epochs=EPOCHS,
    batch_size=BATCH_SIZE, run_name="experiment_lr_1e5"
)
baseline_results.append({"experiment": "Experiment 2: learning rate 1e-5", "pearson": exp2_score})

In [ ]:
results_df = pd.DataFrame(baseline_results)
display(results_df.sort_values("pearson", ascending=False))
results_df.to_csv(f"{OUTPUT_DIR}/experiment_results.csv", index=False)

## 11. Choose the best validation model

Select the best run based on validation Pearson correlation. The model object for each run remains in memory. If you restarted the runtime, rerun the corresponding training cell.

In [ ]:
candidates = [
    ("main", trainer, main_val_pred, main_pearson),
    ("max_len_96", exp1_trainer, exp1_pred, exp1_score),
    ("lr_1e5", exp2_trainer, exp2_pred, exp2_score),
]
best_name, best_trainer, best_val_pred, best_pearson = max(
    candidates, key=lambda item: item[3] if np.isfinite(item[3]) else -np.inf
)
print("Best model:", best_name, "| Validation Pearson:", round(best_pearson, 4))

## 12. Error analysis

Inspect the largest absolute prediction errors. This can reveal difficult pairs, ambiguous technical terms, and cases where lexical similarity does not reflect patent meaning.

In [ ]:
val_analysis = val[["id", "anchor", "target", "context", "score"]].copy()
val_analysis["pred"] = best_val_pred
val_analysis["absolute_error"] = (val_analysis["score"] - val_analysis["pred"]).abs()
val_analysis["error"] = val_analysis["score"] - val_analysis["pred"]
display(val_analysis.sort_values("absolute_error", ascending=False).head(20))
val_analysis.to_csv(f"{OUTPUT_DIR}/val_predictions.csv", index=False)

print("Mean absolute error:", val_analysis["absolute_error"].mean())
print("Validation Pearson:", safe_pearson(val_analysis["score"], val_analysis["pred"]))

### Error-analysis observations (complete after running)

Use the actual rows printed above and write 3–5 observations. Do not claim specific patterns unless they appear in your output.

- Large errors occur when: **[describe examples from the table]**.
- Some phrase pairs may share words but have different technical meanings.
- Short phrases can be ambiguous without their CPC context.
- The model may underpredict or overpredict some score ranges; check the `error` column.

## 13. Predict the test set and create `submission.csv`

The competition submission requires exactly two columns: `id` and `score`. Predictions are clipped to the valid label range `[0, 1]`.

In [ ]:
best_test_ds = to_dataset(test, max_len=96 if best_name == "max_len_96" else MAX_LEN,
                          with_labels=False)
test_predictions = best_trainer.predict(best_test_ds).predictions.reshape(-1)
submission = pd.DataFrame({
    "id": test["id"],
    "score": np.clip(test_predictions, 0.0, 1.0)
})
assert list(submission.columns) == ["id", "score"]
assert len(submission) == len(test)
assert submission["score"].notna().all()
submission.to_csv(f"{OUTPUT_DIR}/submission.csv", index=False)
display(submission.head())
print("Saved:", f"{OUTPUT_DIR}/submission.csv")

## 14. Save model and download outputs

Save the trained model checkpoint and the CSV files to your computer. For a Kaggle offline submission, also make the pretrained checkpoint available as a Kaggle notebook output/dataset input as required by the competition instructions.

In [ ]:
best_trainer.save_model(f"{OUTPUT_DIR}/best_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/best_model")
print("Output files:", os.listdir(OUTPUT_DIR))

In [ ]:
from google.colab import files
files.download(f"{OUTPUT_DIR}/submission.csv")
files.download(f"{OUTPUT_DIR}/val_predictions.csv")
files.download(f"{OUTPUT_DIR}/experiment_results.csv")

## 15. Written answers / report

**1. Why split by anchor?**  
All rows belonging to one anchor are kept in the same split. This prevents the same anchor from appearing in both training and validation data and producing an overly optimistic validation score.

**2. Why use a cross-encoder?**  
A cross-encoder processes both phrases together. Its attention layers can directly compare words and their context across the anchor and target, which is useful for phrase-pair similarity.

**3. Why include CPC context?**  
The same technical phrase can have different meanings in different patent fields. CPC section information gives the model a broad technology-domain clue.

**4. Why use regression?**  
The target score is continuous and can take values 0, 0.25, 0.5, 0.75, and 1. A single output neuron can predict a continuous value and mean squared error can train it.

**5. What is Pearson correlation?**  
Pearson correlation measures the strength and direction of the linear relationship between predicted and actual scores. A value closer to 1 indicates stronger positive correlation.

**6. Why compare against a baseline?**  
A baseline provides a simple reference point. The fine-tuned Transformer is useful only if it improves on a simpler method under the same validation split.

**7. What did the experiments test?**  
Experiment 1 changes maximum sequence length while keeping the other settings fixed. Experiment 2 changes learning rate while keeping the other settings fixed. Compare their validation Pearson scores in `experiment_results.csv`.

**8. What are limitations?**  
The model can struggle with rare technical vocabulary, ambiguous phrases, and domain-specific meanings. Validation performance may also vary with the split, training settings, and available GPU resources.

### Results to fill in after running

- TF-IDF baseline Pearson: **[copy output]**
- Main DeBERTa Pearson: **[copy output]**
- Experiment 1 Pearson: **[copy output]**
- Experiment 2 Pearson: **[copy output]**
- Best configuration: **[copy output]**
- Public leaderboard score: **[fill after Kaggle submission]**
- Private leaderboard score: **[fill after competition closes]**

## 16. Final checklist

- [ ] Both `train.csv` and `test.csv` were loaded successfully.
- [ ] Baseline Pearson score was recorded.
- [ ] Main model trained and validation score was recorded.
- [ ] Both controlled experiments were completed and compared.
- [ ] Error analysis was written using actual validation examples.
- [ ] `submission.csv` was generated and checked.
- [ ] Kaggle submission was made, if required by the competition.
- [ ] Public/private leaderboard scores were recorded when available.

**Important:** the uploaded `project_brief.md` was not provided. If it contains additional requirements beyond the tasks in the starter notebook, add those before final submission.